# Notebook 01 — Foco Antioquia: Deslizamientos e Integración SIATA

**Objetivo:** Acotar el experimento al departamento de Antioquia, filtrar exclusivamente
eventos de remoción en masa, construir el target multiclase y preparar el dataset
para modelado.

**Configuración:** `configs/antioquia_deslizamientos.yaml`
**Continuación de:** `00_experimento_correlacion.ipynb`

> Todos los parámetros de alcance
> (departamento, umbrales, keywords, período) se lee del archivo YAML y se tienen en cuenta para
> registrar íntegramente en MLflow.

## 0. Entorno y configuración

In [1]:
import sys
from pathlib import Path

# Resolución del project root
_cwd = Path().resolve()
for _p in [_cwd, *_cwd.parents]:
    if (_p / "pyproject.toml").exists():
        _root = _p
        break
else:
    _root = _cwd.parent

sys.path.insert(0, str(_root / "src"))

import warnings
warnings.filterwarnings("ignore")

import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from experiment.config import load_config
from experiment.download import load_ideam, load_ungrd
from experiment.process import clean_ideam

plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["font.size"] = 11
sns.set_theme(style="whitegrid")

# ── Carga de configuración ───────────────────────────────────────────────────
cfg = load_config(project_root=_root)

print(" Configuración cargada: configs/antioquia_deslizamientos.yaml")
print(f"  Departamento : {cfg.geo.departamento}")
print(f"  Período      : {cfg.periodo.anio_inicio} – {cfg.periodo.anio_fin}")
print(f"  Umbral medio : n_deslizamientos >= {cfg.target.umbral_medio}")
print(f"  Umbral alto  : n_deslizamientos >= {cfg.target.umbral_alto}")
print(f"  SIATA activo : {cfg.fuentes.siata.activo}")
print(f"  Features     : {len(cfg.all_features)} ({', '.join(cfg.all_features)})")
print(f"  MLflow URI   : {cfg.mlflow_tracking_uri}")

ModuleNotFoundError: No module named 'mlflow'

## 1. Carga de datos crudos

In [ ]:
df_ideam_raw = load_ideam(
    anio_inicio=cfg.periodo.anio_inicio,
    anio_fin=cfg.periodo.anio_fin,
)
df_ungrd_raw = load_ungrd()

print(f"IDEAM crudo : {df_ideam_raw.shape}")
print(f"UNGRD crudo : {df_ungrd_raw.shape}")
print(f"\nColumnas UNGRD: {list(df_ungrd_raw.columns)}")

## 2. Limpieza y filtro geográfico — Antioquia

In [ ]:
def _normalize(s: str) -> str:
    """Minúsculas sin tildes — consistente con process.py."""
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFD", s.lower())
    return "".join(c for c in s if unicodedata.category(c) != "Mn")

df_ideam_clean = clean_ideam(df_ideam_raw)

df_ideam = df_ideam_clean[
    df_ideam_clean["departamento"].apply(_normalize) == cfg.geo.departamento
].copy()

print(f"IDEAM Colombia  : {len(df_ideam_clean):,} registros")
print(f"IDEAM {cfg.geo.departamento.title()} : {len(df_ideam):,} ({len(df_ideam)/len(df_ideam_clean):.1%})")
print(f"Rango temporal  : {df_ideam['fecha'].min().date()} → {df_ideam['fecha'].max().date()}")

## 3. Filtro UNGRD — solo deslizamientos

Keywords de remoción en masa leídas del config (`eventos.landslide_keywords`).
No se hardcodean en el notebook.

In [ ]:
# Detectar columnas clave
evento_col = next(
    (c for c in df_ungrd_raw.columns if any(k in c.lower() for k in ["evento", "tipo"])),
    None,
)
dpto_col = next((c for c in df_ungrd_raw.columns if "depart" in c.lower()), None)
date_col = next((c for c in df_ungrd_raw.columns if "fecha"  in c.lower()), None)

print(f"Columna evento       : '{evento_col}'")
print(f"Columna departamento : '{dpto_col}'")
print(f"Columna fecha        : '{date_col}'")

if evento_col:
    print("\nTop 15 tipos de evento (Colombia):")
    display(df_ungrd_raw[evento_col].value_counts().head(15).to_frame("n"))

In [ ]:
landslide_kw = cfg.eventos.landslide_keywords

df_ungrd = df_ungrd_raw.copy()

# 1) Filtro por tipo de evento
if evento_col:
    mask_evento = df_ungrd[evento_col].apply(
        lambda v: any(k in _normalize(str(v)) for k in landslide_kw)
    )
    df_ungrd = df_ungrd[mask_evento].copy()
    print(f"Después del filtro de evento : {len(df_ungrd):,}")

# 2) Filtro por departamento
if dpto_col:
    mask_dpto = df_ungrd[dpto_col].apply(
        lambda v: _normalize(str(v)) == cfg.geo.departamento
    )
    df_ungrd = df_ungrd[mask_dpto].copy()
    print(f"Después del filtro geográfico: {len(df_ungrd):,}")

# 3) Parsear fecha y extraer período
df_ungrd["fecha"]     = pd.to_datetime(df_ungrd[date_col], errors="coerce")
df_ungrd              = df_ungrd.dropna(subset=["fecha"])
df_ungrd["anio"]      = df_ungrd["fecha"].dt.year
df_ungrd["mes"]       = df_ungrd["fecha"].dt.month
df_ungrd["anio_mes"]  = df_ungrd["fecha"].dt.to_period("M")
df_ungrd["tipo_evento"] = df_ungrd[evento_col] if evento_col else "deslizamiento"

print(f"\n✓ UNGRD {cfg.geo.departamento.title()} (deslizamientos): {len(df_ungrd):,} eventos")
print(f"  Período: {df_ungrd['fecha'].min().date()} → {df_ungrd['fecha'].max().date()}")
print("\nTipos de evento capturados:")
display(df_ungrd["tipo_evento"].value_counts().to_frame("n"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

anual = df_ungrd.groupby("anio").size()
axes[0].bar(anual.index, anual.values, color="#c0392b", edgecolor="white")
axes[0].set_title(f"Deslizamientos por año — {cfg.geo.departamento.title()} (UNGRD)")
axes[0].set_xlabel("Año")
axes[0].set_ylabel("Conteo")

mensual = df_ungrd.groupby("mes").size()
meses = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
axes[1].bar(meses, [mensual.get(i+1, 0) for i in range(12)], color="#c0392b", edgecolor="white")
axes[1].set_title(f"Estacionalidad de deslizamientos — {cfg.geo.departamento.title()}")
axes[1].set_xlabel("Mes")

plt.tight_layout()
plt.show()

## 4. Agregación mensual y target multiclase

Los umbrales se leen del config:
- **Bajo (0):** `n_deslizamientos == 0`
- **Medio (1):** `1 ≤ n_deslizamientos < umbral_alto`
- **Alto (2):** `n_deslizamientos ≥ umbral_alto`

In [ ]:
# Precipitación mensual
precip_monthly = (
    df_ideam
    .groupby("anio_mes", observed=True)
    .agg(
        precip_suma  = ("precip_mm", "sum"),
        precip_max   = ("precip_mm", "max"),
        precip_media = ("precip_mm", "mean"),
        precip_dias  = ("precip_mm", lambda x: (x > 0).sum()),
        n_estaciones = ("precip_mm", "count"),
    )
    .reset_index()
)

# Conteo de deslizamientos por mes
desli_monthly = (
    df_ungrd
    .groupby("anio_mes", observed=True)
    .size()
    .reset_index(name="n_deslizamientos")
)

# Join
df_monthly = precip_monthly.merge(desli_monthly, on="anio_mes", how="left")
df_monthly["n_deslizamientos"] = df_monthly["n_deslizamientos"].fillna(0).astype(int)
df_monthly["anio"] = df_monthly["anio_mes"].dt.year
df_monthly["mes"]  = df_monthly["anio_mes"].dt.month

# Target multiclase — umbrales desde cfg
def asignar_riesgo(n: int) -> int:
    if n == 0:
        return 0
    elif n < cfg.target.umbral_alto:
        return 1
    else:
        return 2

df_monthly["riesgo"]       = df_monthly["n_deslizamientos"].apply(asignar_riesgo)
df_monthly["riesgo_label"] = df_monthly["riesgo"].map({0: "Bajo", 1: "Medio", 2: "Alto"})
df_monthly = df_monthly.sort_values("anio_mes").reset_index(drop=True)

print(f"Dataset mensual: {len(df_monthly):,} meses")
dist = df_monthly["riesgo_label"].value_counts().rename_axis("Riesgo").to_frame("n_meses")
dist["%"] = (dist["n_meses"] / len(df_monthly) * 100).round(1)
display(dist)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

colores = {"Bajo": "#2ecc71", "Medio": "#f39c12", "Alto": "#e74c3c"}
vc = df_monthly["riesgo_label"].value_counts()
axes[0].bar(
    vc.index, vc.values,
    color=[colores.get(l, "gray") for l in vc.index],
    edgecolor="white"
)
axes[0].set_title(
    f"Target multiclase\n"
    f"(medio≥{cfg.target.umbral_medio}, alto≥{cfg.target.umbral_alto})"
)
axes[0].set_xlabel("Nivel de riesgo")
axes[0].set_ylabel("Meses")
for i, (label, val) in enumerate(zip(vc.index, vc.values)):
    axes[0].text(i, val + 0.2, str(val), ha="center", fontsize=11)

ts = df_monthly.copy()
ts.index = ts["anio_mes"].dt.to_timestamp()
axes[1].bar(ts.index, ts["n_deslizamientos"], color="#c0392b", width=20, alpha=0.8, edgecolor="white")
axes[1].axhline(
    cfg.target.umbral_medio, color="#f39c12", linestyle="--", linewidth=1.2,
    label=f"Umbral medio ({cfg.target.umbral_medio})"
)
axes[1].axhline(
    cfg.target.umbral_alto, color="#e74c3c", linestyle="--", linewidth=1.2,
    label=f"Umbral alto ({cfg.target.umbral_alto})"
)
axes[1].set_title(f"Deslizamientos mensuales — {cfg.geo.departamento.title()}")
axes[1].set_ylabel("N° eventos")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Feature engineering

In [ ]:
df = df_monthly.sort_values("anio_mes").copy()

# Rezagos (shift para evitar data leakage)
df["precip_lag1"]   = df["precip_suma"].shift(1)
df["precip_acum3m"] = df["precip_suma"].shift(1).rolling(3, min_periods=1).sum()
df["precip_acum6m"] = df["precip_suma"].shift(1).rolling(6, min_periods=1).sum()
df["precip_max3m"]  = df["precip_max"].shift(1).rolling(3,  min_periods=1).max()
df["precip_dias3m"] = df["precip_dias"].shift(1).rolling(3, min_periods=1).sum()

# Estacionalidad cíclica
df["mes_sin"] = np.sin(2 * np.pi * df["mes"] / 12)
df["mes_cos"] = np.cos(2 * np.pi * df["mes"] / 12)

# Eliminar filas sin historial suficiente (features mínimas definidas en config)
df = df.dropna(subset=cfg.features.required_for_model).reset_index(drop=True)

# Lista final de features activas según config
FEATURES = [f for f in cfg.all_features if f in df.columns]

print(f"Dataset con features: {len(df):,} meses")
print(f"Features activas ({len(FEATURES)}): {FEATURES}")
display(df[FEATURES + ["n_deslizamientos", "riesgo", "riesgo_label"]].describe().round(2))

## 6. Integración SIATA

La activación y la ruta del CSV se controlan desde el config (`fuentes.siata.activo`).
Para activar:
1. Registrarse en https://siata.gov.co → Portal de descargas
2. Descargar series históricas 2019-2022 de precipitación
3. Guardar en `data/raw/siata_precipitacion.csv`
4. Cambiar `fuentes.siata.activo: true` en el YAML y re-ejecutar

In [ ]:
def load_siata(path: Path) -> pd.DataFrame:
    df_s = pd.read_csv(path, low_memory=False)
    print(f"SIATA crudo: {df_s.shape} | columnas: {list(df_s.columns)}")

    date_col_s = next((c for c in df_s.columns if "fecha" in c.lower()), None)
    prec_col_s = next(
        (c for c in df_s.columns if any(k in c.lower() for k in ["precip", "lluvia", "mm"])),
        None,
    )
    if not date_col_s or not prec_col_s:
        raise ValueError(f"CSV SIATA sin columnas esperadas. Detectadas: {list(df_s.columns)}")

    df_s["fecha"]           = pd.to_datetime(df_s[date_col_s], errors="coerce")
    df_s["precip_mm_siata"] = pd.to_numeric(df_s[prec_col_s], errors="coerce")
    df_s = df_s.dropna(subset=["fecha", "precip_mm_siata"])
    df_s = df_s[df_s["precip_mm_siata"] >= cfg.calidad.precip_min_mm]
    df_s["anio_mes"] = df_s["fecha"].dt.to_period("M")

    est_col_s = next(
        (c for c in df_s.columns if any(k in c.lower() for k in ["estacion", "station", "nombre"])),
        None,
    )
    df_s["estacion"] = df_s[est_col_s] if est_col_s else "desconocida"

    print(f"SIATA limpio: {len(df_s):,} | {df_s['fecha'].min().date()} → {df_s['fecha'].max().date()}")
    return df_s[["fecha", "anio_mes", "estacion", "precip_mm_siata"]]


if cfg.fuentes.siata.activo and cfg.siata_csv_path.exists():
    df_siata = load_siata(cfg.siata_csv_path)
    siata_monthly = (
        df_siata
        .groupby("anio_mes", observed=True)
        .agg(
            siata_precip_suma = ("precip_mm_siata", "sum"),
            siata_precip_max  = ("precip_mm_siata", "max"),
            siata_n_registros = ("precip_mm_siata", "count"),
        )
        .reset_index()
    )
    df = df.merge(siata_monthly, on="anio_mes", how="left")
    df["siata_precip_suma"] = df["siata_precip_suma"].fillna(df["precip_suma"])
    df["siata_precip_max"]  = df["siata_precip_max"].fillna(df["precip_max"])
    FEATURES = [f for f in cfg.all_features if f in df.columns]
    print(f"✓ SIATA integrado. Features totales: {len(FEATURES)}")
else:
    activo = cfg.fuentes.siata.activo
    ruta   = cfg.siata_csv_path
    print(f"[SIATA] activo={activo} | archivo={ruta}")
    if not activo:
        print("  → Para activar: cambiar 'fuentes.siata.activo: true' en el YAML")
    else:
        print(f"  → Archivo no encontrado: {ruta}")

## 7. Calidad de datos — cobertura de sensores

In [ ]:
umbral_cobertura = df["n_estaciones"].quantile(
    cfg.calidad.cobertura_sensor_percentil / 100
)

print("=" * 50)
print("DIAGNÓSTICO DE DATOS FALTANTES")
print("=" * 50)

missings = df[FEATURES].isnull().sum()
diag = missings[missings > 0].to_frame("n_missing")
diag["pct"] = (diag["n_missing"] / len(df) * 100).round(1)
if len(diag):
    display(diag)
else:
    print("✓ Sin datos faltantes en el período.")

meses_baja = df[df["n_estaciones"] < umbral_cobertura]
print(
    f"\nMeses con baja cobertura de sensores"
    f" (<p{cfg.calidad.cobertura_sensor_percentil}"
    f" = {umbral_cobertura:.0f} registros): {len(meses_baja)}"
)
if len(meses_baja):
    display(meses_baja[["anio_mes", "n_estaciones", "precip_suma", "riesgo_label"]])

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3))
ts_idx = df["anio_mes"].dt.to_timestamp()
ax.fill_between(ts_idx, df["n_estaciones"], alpha=0.6, color="steelblue")
ax.axhline(
    umbral_cobertura, color="orange", linestyle="--", linewidth=1.2,
    label=f"P{cfg.calidad.cobertura_sensor_percentil} = {umbral_cobertura:.0f}"
)
ax.set_title(f"Cobertura mensual de sensores IDEAM — {cfg.geo.departamento.title()}")
ax.set_ylabel("N° registros")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Persistencia y tracking MLflow

In [ ]:
import shutil

# Guardar dataset procesado
cfg.processed_dir.mkdir(parents=True, exist_ok=True)
output_path = cfg.processed_dir / f"{cfg.geo.departamento}_deslizamientos_v1.csv"
df.to_csv(output_path, index=False)
print(f"✓ Dataset guardado: {output_path.relative_to(_root)}")

# Snapshot del config para trazabilidad
config_snapshot = cfg.processed_dir / "config_snapshot_v1.yaml"
shutil.copy(_root / "configs" / "antioquia_deslizamientos.yaml", config_snapshot)
print(f"✓ Snapshot del config: {config_snapshot.name}")

In [ ]:
mlflow.set_tracking_uri(cfg.mlflow_tracking_uri)
mlflow.set_experiment(cfg.mlflow.experiment_name)

with mlflow.start_run(run_name="dataset_v1", tags=cfg.mlflow.run_tags) as run:

    # Todos los parámetros vienen del config (single source of truth)
    mlflow.log_params(cfg.as_mlflow_params())

    # Métricas del dataset resultante
    mlflow.log_metrics({
        "n_meses"              : len(df),
        "n_deslizamientos"     : int(df["n_deslizamientos"].sum()),
        "pct_riesgo_bajo"      : float((df["riesgo"] == 0).mean()),
        "pct_riesgo_medio"     : float((df["riesgo"] == 1).mean()),
        "pct_riesgo_alto"      : float((df["riesgo"] == 2).mean()),
        "precip_media_mensual" : float(df["precip_suma"].mean()),
        "n_features"           : len(FEATURES),
        "meses_baja_cobertura" : len(meses_baja),
    })

    # Artefactos: dataset + snapshot del config que lo produjo
    mlflow.log_artifact(str(output_path))
    mlflow.log_artifact(str(config_snapshot))

    print(f"✓ MLflow run: {run.info.run_id}")
    print(f"  Experimento : {cfg.mlflow.experiment_name}")
    print(f"  Run         : dataset_v1")
    print(f"  Tags        : {cfg.mlflow.run_tags}")

## 9. Resumen y próximos pasos

### ✓ Completado en este notebook

| Tarea | Estado |
|-------|--------|
| Foco geográfico en Antioquia | ✓ |
| Filtro estricto por deslizamientos (UNGRD) | ✓ |
| Target multiclase (bajo / medio / alto) | ✓ |
| Features con rezago (sin data leakage) | ✓ |
| Diagnóstico de cobertura de sensores | ✓ |
| Módulo de integración SIATA (listo para activar) | ✓ |
| Persistencia + tracking MLflow SQLite | ✓ |
| Sin constantes hardcodeadas — todo desde YAML | ✓ |

### → Notebook 02 — Modelado inicial
- Baseline con `DummyClassifier` (estrategia `stratified`)
- RandomForest multiclase con `class_weight="balanced"`
- Imputación en pipeline de sklearn (`SimpleImputer` / `IterativeImputer`)
- Métricas: F1-macro, accuracy por clase, confusion matrix
- Tracking completo en MLflow (parámetros, métricas, modelo serializado)

### → Notebook 03 — Cruce geoespacial
- Asignación de estaciones a municipios por proximidad (Voronoi / K-nearest)
- Desagregación a nivel municipal con GeoPandas + shapefile DANE

In [ ]:
print("=" * 60)
print("RESUMEN — DATASET v1")
print("=" * 60)
print(f"  Departamento     : {cfg.geo.departamento.title()}")
print(f"  Período          : {cfg.periodo.anio_inicio} – {cfg.periodo.anio_fin}")
print(f"  Meses            : {len(df):,}")
print(f"  Deslizamientos   : {int(df['n_deslizamientos'].sum()):,}")
print(f"\n  Balance multiclase:")
for clase, label in [(0, "Bajo"), (1, "Medio"), (2, "Alto")]:
    n = (df["riesgo"] == clase).sum()
    print(f"    [{clase}] {label:6s}: {n:3d} meses ({n/len(df):.1%})")
print(f"\n  Features         : {len(FEATURES)}")
print(f"  SIATA integrado  : {cfg.fuentes.siata.activo}")
print(f"  Config versionado: configs/antioquia_deslizamientos.yaml")
print(f"  MLflow tracking  : {cfg.mlflow.db_path}")
print("=" * 60)